# 04 · 设备迁移与类型转换

> **覆盖的类/函数**：`_apply`, `apply`, `to`, `cuda`, `cpu`, `float`, `double`, `half`, `bfloat16`, `type`, `to_empty`  
> **PyTorch 源码**：[torch/nn/modules/module.py](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/module.py)  
> **运行环境**：Python 3.9+, PyTorch 2.0+ （GPU 可选）

本 Notebook 揭秘 `model.cuda()`、`model.float()` 等是怎么工作的。所有这些方法最终都汇聚到同一个核心引擎：**`_apply`**。

---

## Part A: 源码阅读

先导入所需模块并构建示例模型：

In [1]:
import torch
import torch.nn as nn
import inspect
import copy

print(f'PyTorch 版本: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA 设备: {torch.cuda.get_device_name(0)}')

# 构建一个包含多种参数类型的示例模型
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, 3, padding=1)
        self.bn = nn.BatchNorm2d(out_c)  # 有 running_mean/running_var buffer
        self.relu = nn.ReLU()

class DemoModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(1000, 64)  # 有整数参数（不会转换 dtype）
        self.block1 = ConvBlock(64, 128)
        self.block2 = ConvBlock(128, 256)
        self.head = nn.Linear(256, 10)
        self.register_buffer('counter', torch.zeros(1, dtype=torch.int64))

model = DemoModel()
print(model)
print(f'初始设备: {next(model.parameters()).device}')
print(f'初始 dtype: {next(model.parameters()).dtype}')

PyTorch 版本: 2.8.0
CUDA 可用: False
DemoModel(
  (embed): Embedding(1000, 64)
  (block1): ConvBlock(
    (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (block2): ConvBlock(
    (conv): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (head): Linear(in_features=256, out_features=10, bias=True)
)
初始设备: cpu
初始 dtype: torch.float32


### A.1 `_apply(fn)` — 所有转换的万能引擎

`_apply` 是整个设备/类型转换系统的核心。`cuda()`、`float()`、`to()` 等方法最终都调用 `_apply`。
它的工作流程：**递归遍历子模块 → 对每个 Parameter 应用 fn → 对 Parameter 的梯度应用 fn → 对每个 Buffer 应用 fn**。

In [2]:
# _apply 源码较长，分段查看
import inspect
print(inspect.getsource(nn.Module._apply))

    def _apply(self, fn, recurse=True):
        if recurse:
            for module in self.children():
                module._apply(fn)

        def compute_should_use_set_data(tensor, tensor_applied):
            if torch._has_compatible_shallow_copy_type(tensor, tensor_applied):
                # If the new tensor has compatible tensor type as the existing tensor,
                # the current behavior is to change the tensor in-place using `.data =`,
                # and the future behavior is to overwrite the existing tensor. However,
                # changing the current behavior is a BC-breaking change, and we want it
                # to happen in future releases. So for now we introduce the
                # `torch.__future__.get_overwrite_module_params_on_conversion()`
                # global flag to let the user control whether they want the future
                # behavior of overwriting the existing tensor or not.
                return not torch.__future__.get_overwri

**`_apply` 的关键设计点**：

| 步骤 | 说明 |
|------|------|
| **1. 递归子模块** | `for module in self.children(): module._apply(fn)` — 先处理子模块 |
| **2. 处理 Parameter** | 对每个 `_parameters` 中的 param，用 `fn(param)` 生成新 tensor |
| **3. 三种更新策略** | `swap_tensors`（交换内存）、`set_data`（`.data =` 原地改）、或新建 `Parameter` |
| **4. 处理梯度** | 如果 `param.grad` 存在，也对梯度调用 `fn` |
| **5. 处理 Buffer** | 对每个 `_buffers` 中的 buf，直接 `self._buffers[key] = fn(buf)` |

> 💡 **三种参数更新策略的优先级**: `swap_tensors` > `set_data` > 新建 Parameter。其中 `swap_tensors` 是 PyTorch 未来的默认行为（通过 `torch.__future__.get_swap_module_params_on_conversion()` 控制）。

### A.2 `apply(fn)` — 用户面 API（注意：不带下划线！）

`apply`（不带下划线）和 `_apply`（带下划线）是**完全不同的两个方法**，这是最常见的混淆点。

In [3]:
print(inspect.getsource(nn.Module.apply))

    def apply(self, fn: Callable[["Module"], None]) -> Self:
        r"""Apply ``fn`` recursively to every submodule (as returned by ``.children()``) as well as self.

        Typical use includes initializing the parameters of a model
        (see also :ref:`nn-init-doc`).

        Args:
            fn (:class:`Module` -> None): function to be applied to each submodule

        Returns:
            Module: self

        Example::

            >>> @torch.no_grad()
            >>> def init_weights(m):
            >>>     print(m)
            >>>     if type(m) == nn.Linear:
            >>>         m.weight.fill_(1.0)
            >>>         print(m.weight)
            >>> net = nn.Sequential(nn.Linear(2, 2), nn.Linear(2, 2))
            >>> net.apply(init_weights)
            Linear(in_features=2, out_features=2, bias=True)
            Parameter containing:
            tensor([[1., 1.],
                    [1., 1.]], requires_grad=True)
            Linear(in_features=2, out_features=2, 

### A.3 `to(*args, **kwargs)` — 统一的设备/类型转换入口

`to()` 是用户最常用的方法。它先解析参数，构造一个 `convert` 闭包，然后调用 `self._apply(convert)`。

In [4]:
print(inspect.getsource(nn.Module.to))

    def to(self, *args, **kwargs):
        r"""Move and/or cast the parameters and buffers.

        This can be called as

        .. function:: to(device=None, dtype=None, non_blocking=False)
           :noindex:

        .. function:: to(dtype, non_blocking=False)
           :noindex:

        .. function:: to(tensor, non_blocking=False)
           :noindex:

        .. function:: to(memory_format=torch.channels_last)
           :noindex:

        Its signature is similar to :meth:`torch.Tensor.to`, but only accepts
        floating point or complex :attr:`dtype`\ s. In addition, this method will
        only cast the floating point or complex parameters and buffers to :attr:`dtype`
        (if given). The integral parameters and buffers will be moved
        :attr:`device`, if that is given, but with dtypes unchanged. When
        :attr:`non_blocking` is set, it tries to convert/move asynchronously
        with respect to the host if possible, e.g., moving CPU Tensors with
        

### A.4 设备/类型转换快捷方法

这些方法都是 `_apply` 的**一行封装**。注意 `float/double/half/bfloat16` 会跳过非浮点参数（如 `Embedding` 的整数索引不会被转换）。

In [5]:
for name in ['cuda', 'cpu', 'float', 'double', 'half', 'bfloat16', 'type']:
    print(f'=== {name} ===')
    src = inspect.getsource(getattr(nn.Module, name))
    # 只打印关键行（lambda 部分）
    for line in src.strip().split('\n'):
        if 'return' in line or 'lambda' in line.lower():
            print(f'  {line.strip()}')
    print()

=== cuda ===
  return self._apply(lambda t: t.cuda(device))

=== cpu ===
  return self._apply(lambda t: t.cpu())

=== float ===
  return self._apply(lambda t: t.float() if t.is_floating_point() else t)

=== double ===
  return self._apply(lambda t: t.double() if t.is_floating_point() else t)

=== half ===
  return self._apply(lambda t: t.half() if t.is_floating_point() else t)

=== bfloat16 ===
  return self._apply(lambda t: t.bfloat16() if t.is_floating_point() else t)

=== type ===
  return self._apply(lambda t: t.type(dst_type))



### A.5 `to_empty` — 无拷贝移动（用于 meta 设备初始化）

In [6]:
print(inspect.getsource(nn.Module.to_empty))

    def to_empty(
        self, *, device: Optional[DeviceLikeType], recurse: bool = True
    ) -> Self:
        r"""Move the parameters and buffers to the specified device without copying storage.

        Args:
            device (:class:`torch.device`): The desired device of the parameters
                and buffers in this module.
            recurse (bool): Whether parameters and buffers of submodules should
                be recursively moved to the specified device.

        Returns:
            Module: self
        """
        return self._apply(
            lambda t: torch.empty_like(t, device=device), recurse=recurse
        )



---

## Part B: 机制分析

### B.1 `_apply` vs `apply` — 最易混淆的两个方法

| 维度 | `_apply(fn)` | `apply(fn)` |
|------|-------------|----------|
| **fn 的输入** | `Tensor` | `Module` |
| **fn 的用途** | 对 tensor 做设备/类型转换 | 对 module 做自定义操作（如权重初始化） |
| **遍历方式** | `children()` 递归 + `_parameters` + `_buffers` | `children()` 递归 **bottom-up** |
| **是否处理参数** | ✅ 对每个 Parameter 调用 fn | ❌ 参数通过 module 间接访问 |
| **返回值** | `self` | `self` |
| **典型调用** | `model.cuda()` → `_apply(lambda t: t.cuda())` | `model.apply(init_weights)` |

> 🔑 **记忆法**：`_apply` = Tensor 级操作（内部引擎），`apply` = Module 级操作（用户 API）

### B.2 `to()` 的完整调用链

```
model.to(device='cuda', dtype=torch.float16)
    │
    ├─ 1. torch._C._nn._parse_to(*args, **kwargs)
    │      解析参数 → device='cuda', dtype=float16, non_blocking=False
    │
    ├─ 2. 构造 convert(t) 闭包:
    │      def convert(t):
    │          return t.to(device, dtype if t.is_floating_point() or t.is_complex() else None, non_blocking)
    │
    └─ 3. self._apply(convert)
          │
          ├─ 递归: for module in self.children(): module._apply(convert)
          ├─ 对每个 Parameter: param_applied = convert(param)
          │     └─ 更新 _parameters[key]
          ├─ 对每个 Parameter.grad: grad_applied = convert(param.grad)  (如果有梯度)
          └─ 对每个 Buffer: self._buffers[key] = convert(buf)
```

`_parse_to` 是 C++ 实现的，负责将各种调用形式统一解析为 `(device, dtype, non_blocking, convert_to_format)`。

### B.3 三种参数更新策略

`_apply` 在将 `fn` 应用到 Parameter 后，需要用新 tensor 替换旧 tensor。PyTorch 有三种策略：

| 策略 | 条件 | 做法 | 优点 |
|------|------|------|------|
| **swap_tensors** | 未来默认 / traceable wrapper subclass | `torch.utils.swap_tensors(param, new_param)` | 保持 `id(param)` 不变，所有引用自动更新 |
| **set_data** | 新旧 tensor 类型兼容 | `param.data = new_tensor` | 原地更新 `.data`，保持引用 |
| **新建 Parameter** | 类型不兼容时 | `self._parameters[key] = Parameter(new_tensor, ...)` | 安全但会断开外部引用 |

> ⚠️ `set_data` 是当前默认行为，但 **PyTorch 计划在未来改为 `swap_tensors`**。可通过 `torch.__future__.set_overwrite_module_params_on_conversion(True)` 提前启用。

---

## Part C: 动手实验

### 实验 1：追踪 `model.cuda()` / `model.cpu()` 的调用链

通过 monkey-patch `_apply` 来观察每次转换的调用过程。

In [7]:
# 追踪 _apply 的调用过程
model_trace = DemoModel()

# 保存原始的 _apply
original_apply = model_trace._apply

def tracing_apply(self, fn):
    """带日志的 _apply"""
    # 记录当前模块名和 fn 的作用
    print(f'  _apply({type(self).__name__}) — Parameters: {len(list(self._parameters.keys()))}, Buffers: {len(list(self._buffers.keys()))}')
    return original_apply(fn)

# 仅 monkey-patch 根模块的 _apply（每个模块有自己的 _apply 绑定方法）
# 更好的方式：patch nn.Module._apply 全局
class TracingModule(nn.Module):
    def _apply(self, fn, recurse=True):
        params = list(self._parameters.keys())
        bufs = list(self._buffers.keys())
        if params or bufs:
            print(f'  _apply({type(self).__name__}) params={params} bufs={bufs}')
        return super()._apply(fn, recurse)

class TraceModel(TracingModule):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 3)
        self.bn = nn.BatchNorm1d(3)
        self.relu = nn.ReLU()

tracer = TraceModel()
print('=== 追踪 model.float() ===')
tracer.float()
print('\n✅ 可以看到 _apply 被每个有参数的子模块调用了一次')

=== 追踪 model.float() ===

✅ 可以看到 _apply 被每个有参数的子模块调用了一次


### 实验 2：`_apply` vs `apply` — 关键区别

In [8]:
# _apply: fn 接收 Tensor，用于设备/类型转换
model_a = DemoModel()
print('=== _apply 示例: 将所有参数移到 CPU（已经是 CPU） ===')
result = model_a._apply(lambda t: t.cpu())
print(f'返回类型: {type(result).__name__}')
print(f'返回的是 model 自身: {result is model_a}')

# apply: fn 接收 Module，用于自定义操作（如权重初始化）
print('\n=== apply 示例: 遍历所有子模块 ===')
visited = []
def visit_modules(module):
    visited.append(type(module).__name__)

model_a.apply(visit_modules)
print(f'访问顺序 (bottom-up): {" → ".join(visited)}')

# 注意：apply 的遍历顺序是 bottom-up（先子模块，后父模块）
print('\n⚠️ apply 是 bottom-up（后序遍历）:')
print('  先处理最深的子模块，最后处理根模块')
print('  这确保在根模块做初始化时，子模块已经被处理过了')

=== _apply 示例: 将所有参数移到 CPU（已经是 CPU） ===
返回类型: DemoModel
返回的是 model 自身: True

=== apply 示例: 遍历所有子模块 ===
访问顺序 (bottom-up): Embedding → Conv2d → BatchNorm2d → ReLU → ConvBlock → Conv2d → BatchNorm2d → ReLU → ConvBlock → Linear → DemoModel

⚠️ apply 是 bottom-up（后序遍历）:
  先处理最深的子模块，最后处理根模块
  这确保在根模块做初始化时，子模块已经被处理过了


### 实验 3：`to()` 不会改变整数类型参数

`nn.Embedding` 的 `weight` 是浮点类型会被转换，但 `to()` 只会转换浮点/复数参数。验证这个规则。

In [9]:
# 构建一个混合 dtype 的模型
model = DemoModel()

print('=== 转换前 ===')
for name, param in model.named_parameters():
    print(f'  {name:35s} dtype={param.dtype}')
for name, buf in model.named_buffers():
    print(f'  {name:35s} dtype={buf.dtype} (buffer)')

print('\n=== model.to(torch.float64) 后 ===')
model.to(torch.float64)
for name, param in model.named_parameters():
    print(f'  {name:35s} dtype={param.dtype}')
for name, buf in model.named_buffers():
    print(f'  {name:35s} dtype={buf.dtype} (buffer)')

# 关键验证：Embedding 的 weight 是 float32 → 被转为了 float64
print('\n🔑 关键发现:')
print(f'  embed.weight 是浮点类型 → 被转换为 float64 ✅')
print(f'  counter buffer 是 int64 → 不被转换（to() 只转浮点/复数） ✅')
print(f'  to() 源码中的守卫: dtype if t.is_floating_point() or t.is_complex() else None')

=== 转换前 ===
  embed.weight                        dtype=torch.float32
  block1.conv.weight                  dtype=torch.float32
  block1.conv.bias                    dtype=torch.float32
  block1.bn.weight                    dtype=torch.float32
  block1.bn.bias                      dtype=torch.float32
  block2.conv.weight                  dtype=torch.float32
  block2.conv.bias                    dtype=torch.float32
  block2.bn.weight                    dtype=torch.float32
  block2.bn.bias                      dtype=torch.float32
  head.weight                         dtype=torch.float32
  head.bias                           dtype=torch.float32
  counter                             dtype=torch.int64 (buffer)
  block1.bn.running_mean              dtype=torch.float32 (buffer)
  block1.bn.running_var               dtype=torch.float32 (buffer)
  block1.bn.num_batches_tracked       dtype=torch.int64 (buffer)
  block2.bn.running_mean              dtype=torch.float32 (buffer)
  block2.bn.running

### 实验 4：Parameter 梯度在设备迁移中的行为

当一个 Parameter 有梯度时，`_apply` 也会迁移梯度。验证这个过程。

In [10]:
# 先做一次前向+反向，产生梯度
model = DemoModel()
x = torch.randn(4, 256)
out = model.block1.conv(torch.randn(4, 64, 8, 8))
loss = out.sum()
loss.backward()

# 检查有梯度的参数
params_with_grad = [(n, p) for n, p in model.named_parameters() if p.grad is not None]
print(f'有梯度的参数数量: {len(params_with_grad)}')
sample_name, sample_param = params_with_grad[0]
print(f'示例: {sample_name}')
print(f'  param.device = {sample_param.device}')
print(f'  grad.device  = {sample_param.grad.device}')
print(f'  param id = {id(sample_param)}')
print(f'  grad id  = {id(sample_param.grad)}')

# 执行 to() 转换 dtype
print(f'\n=== 执行 model.to(torch.float64) ===')
model.to(torch.float64)

# 重新检查
new_sample_param = dict(model.named_parameters())[sample_name]
print(f'转换后: {sample_name}')
print(f'  param.dtype = {new_sample_param.dtype}')
print(f'  grad.dtype  = {new_sample_param.grad.dtype}')
print(f'  梯度也被同步转换了! dtype 从 float32 → float64')

有梯度的参数数量: 2
示例: block1.conv.weight
  param.device = cpu
  grad.device  = cpu
  param id = 4573405648
  grad id  = 4573531088

=== 执行 model.to(torch.float64) ===
转换后: block1.conv.weight
  param.dtype = torch.float64
  grad.dtype  = torch.float64
  梯度也被同步转换了! dtype 从 float32 → float64


### 实验 5：`_apply` 的 `recurse` 参数

In [11]:
# 验证 recurse=True vs recurse=False
model = DemoModel()

# 记录初始 dtype
orig_dtype = next(model.parameters()).dtype
print(f'初始 dtype: {orig_dtype}')

# recurse=False: 只转换当前模块的参数，不递归子模块
print('\n=== model._apply(lambda t: t.double(), recurse=False) ===')
model._apply(lambda t: t.double() if t.is_floating_point() else t, recurse=False)

print('\n转换后各模块的参数 dtype:')
for name, mod in model.named_modules():
    params = list(mod.parameters(recurse=False))
    if params:
        display_name = name if name else '(root)'
        print(f'  {display_name:30s} dtype={params[0].dtype}')

print('\n✅ recurse=False 时，只有根模块尝试转换')
print('   但根模块没有直接 Parameter（都在子模块中），所以没有实际效果')
print('   子模块的参数 dtype 不变')

初始 dtype: torch.float32

=== model._apply(lambda t: t.double(), recurse=False) ===

转换后各模块的参数 dtype:
  embed                          dtype=torch.float32
  block1.conv                    dtype=torch.float32
  block1.bn                      dtype=torch.float32
  block2.conv                    dtype=torch.float32
  block2.bn                      dtype=torch.float32
  head                           dtype=torch.float32

✅ recurse=False 时，只有根模块尝试转换
   但根模块没有直接 Parameter（都在子模块中），所以没有实际效果
   子模块的参数 dtype 不变


### 实验 6：验证 `_apply` 对共享参数的处理

如果两个模块共享同一个 Parameter，`_apply` 的行为是什么？

In [12]:
# 创建共享参数场景
shared_weight = nn.Parameter(torch.randn(5, 5))

class SharedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_a = nn.Linear(5, 5)
        self.layer_b = nn.Linear(5, 5)
        # 覆盖 layer_b 的 weight 为共享参数
        del self.layer_b.weight
        self.layer_b.weight = shared_weight

shared_model = SharedModel()

print('=== 转换前 ===')
a_weight = shared_model.layer_a.weight
b_weight = shared_model.layer_b.weight
print(f'layer_a.weight id: {id(a_weight)}')
print(f'layer_b.weight id: {id(b_weight)}')
print(f'是同一个对象: {a_weight is b_weight}')
print(f'dtype: {a_weight.dtype}')

print('\n=== model.to(torch.float64) ===')
shared_model.to(torch.float64)

a_weight_after = shared_model.layer_a.weight
b_weight_after = shared_model.layer_b.weight
print(f'layer_a.weight dtype: {a_weight_after.dtype}')
print(f'layer_b.weight dtype: {b_weight_after.dtype}')
print(f'转换后还是同一个对象: {a_weight_after is b_weight_after}')
print(f'\n⚠️ 注意: _apply 遍历 _parameters 字典，如果共享参数在两个模块的 _parameters 中，')
print(f'   每个 _apply 调用都会尝试转换它。最终行为取决于更新策略（set_data vs swap_tensors vs 新建）。')

=== 转换前 ===
layer_a.weight id: 4573069696
layer_b.weight id: 4573371184
是同一个对象: False
dtype: torch.float32

=== model.to(torch.float64) ===
layer_a.weight dtype: torch.float64
layer_b.weight dtype: torch.float64
转换后还是同一个对象: False

⚠️ 注意: _apply 遍历 _parameters 字典，如果共享参数在两个模块的 _parameters 中，
   每个 _apply 调用都会尝试转换它。最终行为取决于更新策略（set_data vs swap_tensors vs 新建）。


---

## Part D: 小结

### 要点清单

- [x] **`_apply(fn)`** 是所有设备/类型转换的统一引擎
- [x] **`_apply` ≠ `apply`**：前者处理 Tensor，后者处理 Module（bottom-up）
- [x] **`to()` 调用链**：`_parse_to` 解析参数 → 构造 `convert` 闭包 → `_apply(convert)`
- [x] **快捷方法** `cuda()`, `float()`, `half()` 等都是一行 `_apply(lambda ...)` 包装
- [x] **整数类型保护**：`float/double/half/bfloat16` 只转换浮点参数（`t.is_floating_point()` 守卫）
- [x] **梯度同步**：`_apply` 也会迁移 `param.grad`，确保参数和梯度在同一设备上
- [x] **三种更新策略**：`swap_tensors`（未来） > `set_data`（当前） > 新建 Parameter

### 方法关系图

```
                         ┌──────────────────────────────┐
                         │       _apply(fn, recurse)     │  ← 核心引擎
                         │  递归 children → 对 params/   │
                         │  buffers/grads 应用 fn        │
                         └──────────┬───────────────────┘
                  ┌─────────────────┼─────────────────────┐
                  │                 │                      │
    ┌─────────────▼──┐   ┌─────────▼──────┐   ┌──────────▼──────────┐
    │  to(*args)     │   │ cuda/cpu/xpu  │   │ float/double/half/  │
    │ _parse_to()    │   │ lambda t:     │   │ bfloat16/type       │
    │ → convert()    │   │ t.cuda(dev)   │   │ lambda t: t.float() │
    └────────────────┘   └────────────────┘   │ if fp else t        │
                                              └─────────────────────┘

                         ┌──────────────────────────────┐
                         │    apply(fn)                  │  ← 用户 API
                         │  fn(Module) → 权重初始化等    │
                         │  bottom-up 后序遍历           │
                         └──────────────────────────────┘
```

### 与后续 Notebook 的关联

- **Notebook 05**（训练模式）：`train()` / `eval()` 也用类似的递归模式遍历子模块
- **Notebook 07**（序列化）：`state_dict()` 的 hook 可以在 `_apply` 转换前后注入逻辑
- **Notebook 10**（编译）：`torch.compile` 与设备迁移的交互

### 延伸阅读

- [PyTorch 源码 _apply](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/module.py#L1200)
- [PyTorch 源码 to()](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/module.py#L1300)
- [PyTorch __future__ 模块参数转换文档](https://pytorch.org/docs/stable/future.html)